# Hafta 4 — Dağıtık Sözcük Temsilleri

Seyrek sayımlardan yoğun vektörlere. word2vec'i (negatif örneklemeli skip-gram), GloVe'u ve fastText'i her birini NumPy ile sıfırdan uyguluyoruz.

## Öğrenme Hedefleri

- Negatif örneklemeli skip-gram (SGNS) hedefini türetmek ve bunun kaydırılmış bir PMI matrisini çarpanlarına ayırmaya eşdeğer olduğunu kanıtlamak (Levy ve Goldberg, 2014).
- SGNS'yi elle gradyan hesaplayarak NumPy ile uygulamak.
- GloVe'u uygulamak ve hedefinin neden ağırlıklı en küçük kareler matris çarpanlarına ayırma olduğunu açıklamak.
- Gömmeleri içsel (analoji, benzerlik) ve dışsal (sınıflandırma) olarak değerlendirmek.

## Zorunlu Okumalar

- Mikolov, T. vd. (2013). *Distributed Representations of Words and Phrases and Their Compositionality*.
- Pennington, J., Socher, R. ve Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*.
- Levy, O. ve Goldberg, Y. (2014). *Neural Word Embedding as Implicit Matrix Factorization*.
- Bojanowski, P. vd. (2017). *Enriching Word Vectors with Subword Information*.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

## 1. Dağılımsal hipotez

Firth (1957): *"Bir sözcüğü, birlikte bulunduğu sözcüklerden tanırsınız."* (*"You shall know a word by the company it keeps."*) Bu, tüm dağılımsal anlambilimin temel varsayımıdır. Onu, benzer bağlamlarda görünen sözcüklerin benzer vektörlere sahip olacağı bir $w \mapsto \mathbf{v}_w \in \mathbb{R}^d$ eşlemesi öğrenerek işlevsel hâle getiriyoruz.

## 2. Negatif örneklemeli skip-gram (SGNS)

Bir merkez sözcük $w$, gerçek bir bağlam sözcüğü $c$ ve bir gürültü dağılımı $P_n$'den çekilen $k$ negatif örnek $c'_1, \ldots, c'_k$ için:

$$\mathcal{L}(w, c) = \log \sigma(\mathbf{v}_c^\top \mathbf{v}_w) + \sum_{i=1}^{k} \log \sigma(-\mathbf{v}_{c'_i}^\top \mathbf{v}_w).$$

Bunu en büyüklemek şu anlama gelir: gerçek (sözcük, bağlam) çiftini yüksek nokta çarpımına, rastgele çiftleri düşük nokta çarpımına it. Gürültü dağılımı, 3/4 kuvvetine yükseltilmiş unigram dağılımıdır (Mikolov vd., 2013); bu, çok sık geçen sözcüklerin ağırlığını azaltır.

**Gradyanlar** (elle hesaplanmış; $\mathcal{L}$ en büyüklendiği için bunlar gradyan *yükselişi* yönleridir — kod, kaybın negatifi üzerinde gradyan inişi yapar):

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_w} = (1 - \sigma(\mathbf{v}_c^\top \mathbf{v}_w)) \mathbf{v}_c - \sum_i \sigma(\mathbf{v}_{c'_i}^\top \mathbf{v}_w) \mathbf{v}_{c'_i}.$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_c} = (1 - \sigma(\mathbf{v}_c^\top \mathbf{v}_w)) \mathbf{v}_w.$$

In [ ]:
class SGNS:
    def __init__(self, vocab_size, dim=64, window=2, neg=5, lr=0.025):
        self.V, self.dim, self.window, self.neg, self.lr = vocab_size, dim, window, neg, lr
        # İki gömme matrisi: 'giriş' (merkez) ve 'çıkış' (bağlam).
        self.W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.C = np.zeros((vocab_size, dim))

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def _build_sampling_table(self, counts, size=10**6):
        # Unigram^(3/4) örnekleme tablosu (Mikolov hilesi).
        probs = np.array([counts.get(i, 0) for i in range(self.V)], dtype=float) ** 0.75
        probs /= probs.sum()
        table = np.random.choice(self.V, size=size, p=probs)
        return table

    def train(self, sequences, epochs=5, verbose=True):
        counts = Counter(tok for seq in sequences for tok in seq)
        self.neg_table = self._build_sampling_table(counts)

        for ep in range(epochs):
            losses = []
            for seq in sequences:
                for i, w in enumerate(seq):
                    start, end = max(0, i - self.window), min(len(seq), i + self.window + 1)
                    for j in range(start, end):
                        if j == i:
                            continue
                        c = seq[j]
                        # Pozitif örnek
                        score_pos = self.W[w] @ self.C[c]
                        sig_pos = self._sigmoid(score_pos)
                        grad_w_acc = (sig_pos - 1) * self.C[c]
                        self.C[c] -= self.lr * (sig_pos - 1) * self.W[w]

                        # Negatif örnekler
                        negs = self.neg_table[np.random.randint(0, len(self.neg_table), size=self.neg)]
                        for c_neg in negs:
                            if c_neg == c:
                                continue
                            score_neg = self.W[w] @ self.C[c_neg]
                            sig_neg = self._sigmoid(score_neg)
                            grad_w_acc += sig_neg * self.C[c_neg]
                            self.C[c_neg] -= self.lr * sig_neg * self.W[w]

                        self.W[w] -= self.lr * grad_w_acc
                        losses.append(-np.log(sig_pos + 1e-12))
            if verbose:
                print(f"dönem {ep+1}/{epochs}  kayıp = {np.mean(losses):.4f}")
        return self

# Belirgin anlamsal gruplara sahip küçük bir derlem kur (örnek cümleler İngilizcedir).
SENTENCES = [
    "the cat sat on the mat", "the dog sat on the rug", "the cat chased the mouse",
    "the dog chased the cat", "the king ruled the kingdom", "the queen ruled the kingdom",
    "the prince met the princess", "the queen met the king",
    "machine learning models learn from data", "deep learning models process language",
    "neural networks learn representations from data", "language models predict the next word",
    "transformers process sequences in parallel", "attention models long range dependencies",
] * 8

tokenized = [tokenize(s) for s in SENTENCES]
vocab = sorted(set(t for s in tokenized for t in s))
tok2id = {t: i for i, t in enumerate(vocab)}
id2tok = {i: t for t, i in tok2id.items()}
sequences = [[tok2id[t] for t in s] for s in tokenized]
print(f"|V| = {len(vocab)}, |cümle| = {len(sequences)}")

model = SGNS(vocab_size=len(vocab), dim=32, window=2, neg=5, lr=0.05)
model.train(sequences, epochs=20, verbose=False)
print("Eğitim tamamlandı.")

In [ ]:
# Gömme uzayında en yakın komşular — niteliksel bir akıl sağlığı denetimi.
def cosine_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

def nearest(model, word, k=5):
    if word not in tok2id:
        return []
    v = model.W[tok2id[word]]
    sims = [(t, cosine_sim(v, model.W[i])) for t, i in tok2id.items() if t != word]
    return sorted(sims, key=lambda x: -x[1])[:k]

for q in ['cat', 'king', 'learning', 'transformers']:
    print(f"{q:>14}: {[(w, f'{s:.2f}') for w, s in nearest(model, q, 4)]}")

## 3. Levy ve Goldberg: SGNS kaydırılmış PMI'yi çarpanlarına ayırır

Levy ve Goldberg (2014), yeterli gömme boyutuyla SGNS hedefinin şu durumda en iyilendiğini kanıtladı:

$$\mathbf{v}_c^\top \mathbf{v}_w = \text{PMI}(w, c) - \log k.$$

Bu, sinirsel sözcük gömmelerini klasik dağılımsal anlambilimin sayım tabanlı PMI'sine geri bağlar. Aşağıda deneysel kaydırılmış PMI matrisini kuruyor ve SGNS nokta çarpımlarımızla korelasyonunu doğruluyoruz.

In [ ]:
def empirical_pmi(sequences, V, window=2, k=5):
    word_counts = Counter()
    pair_counts = Counter()
    total_pairs = 0
    for seq in sequences:
        for i, w in enumerate(seq):
            word_counts[w] += 1
            for j in range(max(0, i-window), min(len(seq), i+window+1)):
                if j != i:
                    pair_counts[(w, seq[j])] += 1
                    total_pairs += 1
    # PMI(w,c) = log p(w,c) / (p(w) p(c)), log k kadar kaydırılmış.
    total_words = sum(word_counts.values())
    pmi = np.full((V, V), -np.inf)
    for (w, c), cnt in pair_counts.items():
        p_wc = cnt / total_pairs
        p_w = word_counts[w] / total_words
        p_c = word_counts[c] / total_words
        pmi[w, c] = np.log(p_wc / (p_w * p_c)) - np.log(k)
    return pmi

shifted_pmi = empirical_pmi(sequences, V=len(vocab), window=2, k=5)
sgns_dot = model.W @ model.C.T

# SGNS nokta çarpımı ile kaydırılmış PMI arasındaki korelasyon (yalnızca gözlenen çiftlerde).
mask = np.isfinite(shifted_pmi)
corr = np.corrcoef(sgns_dot[mask], shifted_pmi[mask])[0, 1]
print(f"Gözlenen çiftlerde korelasyon(SGNS nokta çarpımı, kaydırılmış PMI): {corr:.3f}")
print("(Daha uzun eğitim / daha büyük boyutla artar — Levy ve Goldberg 1'e yaklaştığını gösterir.)")

## 4. GloVe — ağırlıklı en küçük kareler matris çarpanlarına ayırma

Pennington vd. (2014) eş-oluşum oranlarından yola çıkarak şunu türetir:

$$J = \sum_{i,j} f(X_{ij}) (\mathbf{v}_i^\top \mathbf{v}_j + b_i + b_j - \log X_{ij})^2,$$

burada $X_{ij}$, (i, j) eş-oluşum sayısıdır ve ağırlık fonksiyonu

$$f(x) = \min((x / x_{\max})^\alpha, 1)$$

çok sık geçen çiftlerin ağırlığını azaltır. Bu, SGNS'nin örtük olarak yaptığı matris çarpanlarına ayırmanın açık hâlidir.

In [ ]:
class GloVe:
    def __init__(self, vocab_size, dim=32, x_max=10, alpha=0.75, lr=0.05):
        self.V, self.dim, self.x_max, self.alpha, self.lr = vocab_size, dim, x_max, alpha, lr
        self.W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.W_tilde = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.b = np.zeros(vocab_size); self.b_tilde = np.zeros(vocab_size)

    def _cooccur(self, sequences, window=2):
        X = np.zeros((self.V, self.V))
        for seq in sequences:
            for i, w in enumerate(seq):
                for j in range(max(0, i-window), min(len(seq), i+window+1)):
                    if i != j:
                        X[w, seq[j]] += 1.0 / abs(i - j)  # uzaklığa göre ağırlıklı (GloVe kuralı)
        return X

    def _f(self, x):
        return np.where(x < self.x_max, (x / self.x_max) ** self.alpha, 1.0)

    def train(self, sequences, epochs=50, verbose=False):
        X = self._cooccur(sequences)
        pairs = np.argwhere(X > 0)
        for ep in range(epochs):
            losses = []
            np.random.shuffle(pairs)
            for i, j in pairs:
                xij = X[i, j]
                pred = self.W[i] @ self.W_tilde[j] + self.b[i] + self.b_tilde[j]
                diff = pred - np.log(xij)
                weight = self._f(xij)
                grad = 2 * weight * diff
                self.W[i]       -= self.lr * grad * self.W_tilde[j]
                self.W_tilde[j] -= self.lr * grad * self.W[i]
                self.b[i]       -= self.lr * grad
                self.b_tilde[j] -= self.lr * grad
                losses.append(weight * diff * diff)
            if verbose and ep % 10 == 0:
                print(f"dönem {ep:3d}  kayıp = {np.mean(losses):.4f}")
        return self

    def vectors(self):
        # Nihai gömme = ikisinin toplamı (standart GloVe kuralı).
        return self.W + self.W_tilde

glove = GloVe(vocab_size=len(vocab), dim=32, x_max=10).train(sequences, epochs=80)
G = glove.vectors()
print("GloVe en yakın komşular:")
for q in ['cat', 'king', 'learning']:
    if q in tok2id:
        v = G[tok2id[q]]
        sims = sorted([(t, cosine_sim(v, G[i])) for t, i in tok2id.items() if t != q],
                      key=lambda x: -x[1])[:4]
        print(f"  {q:>14}: {[(w, f'{s:.2f}') for w, s in sims]}")

## 5. fastText — alt sözcük gömmeleri

Bojanowski vd. (2017) her sözcüğü, karakter n-gram gömmelerinin toplamı olarak temsil eder:

$$\mathbf{v}_w = \sum_{g \in G_w} \mathbf{z}_g,$$

burada $G_w$, $w$'nin karakter n-gramlarının (genellikle $n \in [3, 6]$) ve $w$'nin kendisinin kümesidir. Bu, sözlük dışı sözcüklere ve Türkçe gibi biçimbilimsel olarak zengin dillere doğal biçimde genellenir.

In [ ]:
def char_ngrams(word, n_min=3, n_max=6):
    w = f"<{word}>"
    grams = set()
    for n in range(n_min, n_max + 1):
        for i in range(len(w) - n + 1):
            grams.add(w[i:i+n])
    grams.add(word)
    return grams

# Gösterim: biçimbilimsel olarak ilişkili Türkçe sözcükler n-gramları paylaşır,
# bu yüzden eğitimden önce bile gömme kütlesini paylaşırlar.
for w in ['ev', 'evler', 'evimde', 'evlerimde']:
    g = sorted(char_ngrams(w, 3, 5))
    print(f"{w:>12}: {len(g)} n-gram, örnek: {g[:6]}")

## 6. İçsel değerlendirme: benzerlik ve analoji

İki standart aile:

- **Benzerlik** — (sözcük_a, sözcük_b, insan_puanı) verildiğinde, kosinüs benzerliği ile insan yargısı arasındaki Spearman korelasyonunu ölç (WordSim-353, SimLex-999).
- **Analoji** — *man is to woman as king is to ?* ("erkek kadına neyse, kral neye?") sorusu $\arg\max_w \cos(\mathbf{v}_w, \mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}})$ ile çözülür; beklenen yanıt *queen*'dir (Mikolov vd., 2013). Aşağıdaki `solve_analogy(a, b, c)` aynı kalıbı izler: *a, b'ye neyse; c neye?* → $\mathbf{v}_b - \mathbf{v}_a + \mathbf{v}_c$.

Her iki kıyaslama da kusurludur (Linzen, 2016) ama yararlı teşhis araçları olmayı sürdürür.

In [ ]:
def solve_analogy(emb, tok2id, id2tok, a, b, c, top_k=3, exclude=True):
    if not all(w in tok2id for w in (a, b, c)):
        return [(None, 0.0)]
    v = emb[tok2id[b]] - emb[tok2id[a]] + emb[tok2id[c]]
    sims = []
    for t, i in tok2id.items():
        if exclude and t in (a, b, c):
            continue
        sims.append((t, cosine_sim(v, emb[i])))
    return sorted(sims, key=lambda x: -x[1])[:top_k]

print("Analoji: cat:dog :: king:?")
print(" ", solve_analogy(model.W, tok2id, id2tok, 'cat', 'dog', 'king'))
print("\nAnaloji (GloVe): cat:dog :: king:?")
print(" ", solve_analogy(G, tok2id, id2tok, 'cat', 'dog', 'king'))

## 7. Görselleştirme

Öğrenilen gömmeleri PCA ile 2B'ye izdüşürün. Anlamsal kümeler ortaya çıkmalıdır.

In [ ]:
def pca_2d(X):
    Xc = X - X.mean(0, keepdims=True)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

proj = pca_2d(model.W)
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(proj[:, 0], proj[:, 1], s=30, alpha=0.7)
for t, i in tok2id.items():
    ax.annotate(t, proj[i], fontsize=9, xytext=(3, 3), textcoords='offset points')
ax.set(title='SGNS gömmeleri — PCA izdüşümü', xlabel='TB1', ylabel='TB2')
plt.tight_layout(); plt.show()

## 8. Alıştırmalar

1. **Levy ve Goldberg'i yeniden üretin.** Bir derlem üzerinde $d = |V|$ ile SGNS eğitin, ardından ortaya çıkan $\mathbf{W} \mathbf{C}^\top$ matrisinin, optimizasyon gürültüsü sınırları içinde kaydırılmış PMI matrisiyle eşleştiğini doğrulayın.
2. **Türkçe için fastText.** Bir Türkçe derlem üzerinde fastText eğitin. Biçimbilimsel olarak ilişkili sözcük grupları (*ev, evler, evimde, ...*) için en yakın komşu kalitesini word2vec ile karşılaştırın.
3. **Ölçekte analoji.** Google analoji veri kümesini indirin. word2vec/GloVe doğruluğunu her kategori için değerlendirin. Nerede başarısız oluyorlar?
4. **Gömme önyargısı.** Eğittiğiniz gömmelerin cinsiyet önyargısı sergileyip sergilemediğini kontrol edin (Bolukbasi vd., 2016): $\mathbf{v}_{\text{nurse}} - \mathbf{v}_{\text{doctor}}$ ile $\mathbf{v}_{\text{she}} - \mathbf{v}_{\text{he}}$ aynı yönde mi?

---

## Gelecek Hafta

Hafta 5 — Dil Modelleri: n-gramlardan sinirsel modellere. Görevi biçimselleştiriyor, perpleksiteyi tanımlıyor ve Bengio (2003) sinirsel dil modelini uyguluyoruz.